# IgBLAST-аннотация — ERP003950 (мышь)

Гоняет IgBLAST по всем merged (`assemble-pass`) FASTQ мышиного датасета `ERP003950`: FASTQ→FASTA, затем `igblastn` по каждому sample. Resume-safe: завершённые TSV пропускаются, неполные/устаревшие архивируются и пересчитываются. По умолчанию `nproc=4` — на этой задаче `nproc=8` убивал `ERR346598` (OOM).


In [ ]:
# CELL 1: настройка PATH + LD_LIBRARY_PATH для IgBLAST
import os
os.environ["LD_LIBRARY_PATH"] = "/data/user/epishkin/conda/envs/bcr_env/lib"
os.environ["IGDATA"] = "/data/user/epishkin/igblast"
os.environ["PATH"] = (
    "/data/user/epishkin/igblast/bin:"
    + "/data/user/epishkin/conda/envs/bcr_env/bin:"
    + os.environ.get("PATH", "")
)
print("LD_LIBRARY_PATH:", os.environ["LD_LIBRARY_PATH"])
print("IGDATA:", os.environ["IGDATA"])
print("PATH:", os.environ["PATH"].split(":")[0])
print("OK setup")


In [ ]:
# CELL 2: импорты, константы, проверка путей
import subprocess, time, sys
from pathlib import Path

IGDATA   = Path("/data/user/epishkin/igblast")
IGBLASTN = IGDATA / "bin" / "igblastn"

GERMLINE_V = IGDATA / "internal_data" / "mouse" / "mouse_gl_V"
GERMLINE_D = IGDATA / "internal_data" / "mouse" / "mouse_gl_D"
GERMLINE_J = IGDATA / "internal_data" / "mouse" / "mouse_gl_J"
AUX        = IGDATA / "optional_file" / "mouse_gl.aux"

checks = {
    "igblastn": IGBLASTN,
    "V_DB":     GERMLINE_V,
    "V.nhr":    GERMLINE_V.with_suffix(".nhr"),
    "D.nhr":    GERMLINE_D.with_suffix(".nhr"),
    "J.nhr":    GERMLINE_J.with_suffix(".nhr"),
    "AUX":      AUX,
}

all_ok = True
for name, p in checks.items():
    ok = p.exists()
    if not ok:
        all_ok = False
    print(f"  {name:>8s}: {'OK' if ok else 'MISSING'}  {p}")

if all_ok:
    print("\nAll paths OK — ready to annotate")
else:
    print("\nMISSING files — fix and re-run")


In [ ]:
# CELL 3: хелперы — count_fasta + fastq_to_fasta
def count_fasta(path):
    r = subprocess.run(["grep", "-c", "^>", str(path)],
                       capture_output=True, text=True)
    return int(r.stdout.strip())

def fastq_to_fasta(fq_in, fa_out):
    if fa_out.exists() and fa_out.stat().st_size > 0:
        size_mb = fa_out.stat().st_size / 1e6
        n = count_fasta(fa_out)
        print(f"    already exists: {n:,} seqs, {size_mb:.1f} MB")
        return

    t0 = time.time()
    print(f"    FASTQ->FASTA: {fq_in.name}...")

    awk_code = 'NR%4==1{sub("^@",">",$0);print} NR%4==2{print}'
    fa_out.parent.mkdir(parents=True, exist_ok=True)

    with open(fa_out, "w", buffering=64*1024*1024) as fout:
        proc = subprocess.Popen(
            ["bash", "-c", f"zcat {fq_in} | awk '{awk_code}'"],
            stdout=fout,
            stderr=subprocess.PIPE,
            text=True
        )
        while proc.poll() is None:
            elapsed = int(time.time() - t0)
            if fa_out.stat().st_size > 0:
                out_mb = fa_out.stat().st_size / 1e6
            else:
                out_mb = 0
            print(f"      ...{elapsed}s, {out_mb:.1f} MB, PID={proc.pid}")
            time.sleep(30)
        _, stderr = proc.communicate()

    elapsed = (time.time() - t0) / 60
    size_mb = fa_out.stat().st_size / 1e6
    n_seqs = count_fasta(fa_out)
    print(f"    DONE: {n_seqs:,} seqs, {size_mb:.1f} MB in {elapsed:.1f} min")

print("Helpers defined: count_fasta(), fastq_to_fasta()")


In [ ]:
# CELL 4: раннер IgBLAST для одного sample
# Политика безопасного resume:
# - завершённый TSV пропускается, только если line_count-1 близко к числу FASTA-записей
# - частичный/устаревший TSV (например, после убитого запуска) переименовывается в *.stale.* и пересчитывается
# - nproc=4 по умолчанию, потому что nproc=8 убил ERR346598 на этой задаче

def tsv_line_count(path):
    if not Path(path).exists():
        return 0
    r = subprocess.run(["wc", "-l", str(path)], capture_output=True, text=True)
    return int(r.stdout.split()[0]) if r.stdout.strip() else 0

def is_complete_igblast_tsv(out_tsv, expected_records, min_fraction=0.90):
    if not out_tsv.exists() or out_tsv.stat().st_size == 0:
        return False, 0, 0.0
    lines = tsv_line_count(out_tsv)
    records = max(0, lines - 1)  # header + строки аннотации
    fraction = records / expected_records if expected_records else 0.0
    return fraction >= min_fraction, lines, fraction

def archive_stale_output(path, reason):
    path = Path(path)
    if not path.exists():
        return None
    stamp = time.strftime("%Y%m%d_%H%M%S")
    dest = path.with_name(path.name + f".stale_{reason}_{stamp}")
    path.rename(dest)
    return dest

def run_igblast(fa_dir, out_dir, sample, nproc=4, min_complete_fraction=0.90):
    fa_file  = Path(fa_dir) / f"{sample}.fa"
    out_tsv  = Path(out_dir) / f"{sample}_igblast.tsv"
    log_file = Path(out_dir) / f"{sample}_igblast.log"

    Path(out_dir).mkdir(parents=True, exist_ok=True)

    n_seqs = count_fasta(fa_file)
    complete, lines, fraction = is_complete_igblast_tsv(out_tsv, n_seqs, min_complete_fraction)
    if complete:
        size_mb = out_tsv.stat().st_size / 1e6
        print(f"  [{sample}] skip complete ({lines:,} lines, {size_mb:.1f} MB, {fraction:.1%} of FASTA records)")
        return

    if out_tsv.exists() and out_tsv.stat().st_size > 0:
        size_mb = out_tsv.stat().st_size / 1e6
        archived = archive_stale_output(out_tsv, "incomplete")
        print(f"  [{sample}] stale/incomplete output archived: {archived.name} ({lines:,} lines, {size_mb:.1f} MB, {fraction:.1%})")
        if log_file.exists():
            log_archived = archive_stale_output(log_file, "incomplete")
            print(f"  [{sample}] stale log archived: {log_archived.name}")

    print(f"  [{sample}] RUN {n_seqs:,} reads, nproc={nproc}")

    cmd = [
        str(IGBLASTN),
        "-germline_db_V", str(GERMLINE_V),
        "-germline_db_D", str(GERMLINE_D),
        "-germline_db_J", str(GERMLINE_J),
        "-organism",       "mouse",
        "-ig_seqtype",     "Ig",
        "-domain_system",  "imgt",
        "-query",          str(fa_file),
        "-auxiliary_data", str(AUX),
        "-outfmt",         "19",
        "-num_threads",    str(nproc),
        "-out",            str(out_tsv),
    ]

    t0 = time.time()
    sys.stdout.flush()

    with open(log_file, "w") as log:
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, text=True)
        print(f"    PID={proc.pid}; log={log_file}")

        while proc.poll() is None:
            elapsed = int(time.time() - t0)
            out_mb = out_tsv.stat().st_size / 1e6 if out_tsv.exists() else 0.0
            current_lines = tsv_line_count(out_tsv) if out_tsv.exists() else 0
            current_records = max(0, current_lines - 1)
            print(f"    [{sample}] {elapsed}s, {out_mb:.1f} MB, {current_records:,}/{n_seqs:,} rows, PID={proc.pid}")
            sys.stdout.flush()
            time.sleep(120)

        rc = proc.returncode

    elapsed = (time.time() - t0) / 60
    out_mb = out_tsv.stat().st_size / 1e6 if out_tsv.exists() else 0
    n_lines = tsv_line_count(out_tsv)
    n_records = max(0, n_lines - 1)
    fraction = n_records / n_seqs if n_seqs else 0.0

    print(f"    [{sample}] DONE: {elapsed:.1f} min, {n_records:,}/{n_seqs:,} rows ({fraction:.1%}), {out_mb:.1f} MB, rc={rc}")

    if rc != 0:
        raise RuntimeError(f"{sample} igblast failed rc={rc}; check log: {log_file}")
    if fraction < min_complete_fraction:
        raise RuntimeError(f"{sample} igblast output incomplete: {n_records:,}/{n_seqs:,} rows ({fraction:.1%}); check log: {log_file}")

print("Helper defined: run_igblast() with safe stale-output detection, nproc default=4")


In [ ]:
# CELL 5: запуск аннотации для ERP003950 (мышь)
MERGED_DIR  = "/data/user/epishkin/results/ERP003950/merged"
FASTA_DIR   = "/data/user/epishkin/results/ERP003950/igblast/fasta"
IGBLAST_DIR = "/data/user/epishkin/results/ERP003950/igblast"

merged_fq = Path(MERGED_DIR) / "fastq"
SAMPLES = sorted(
    f.name.replace("_assemble-pass.fastq.gz", "")
    for f in merged_fq.glob("*_assemble-pass.fastq.gz")
)

print(f"[ERP003950] {len(SAMPLES)} samples: {SAMPLES}")

# Шаг 1: FASTQ -> FASTA
print("=== Step 1: FASTQ -> FASTA ===")
for s in SAMPLES:
    fq = merged_fq / f"{s}_assemble-pass.fastq.gz"
    fa = Path(FASTA_DIR) / f"{s}.fa"
    fastq_to_fasta(fq, fa)

# Шаг 2: аннотация IgBLAST
print(f"\n=== Step 2: IgBLAST ({len(SAMPLES)} samples) ===")
for s in SAMPLES:
    run_igblast(FASTA_DIR, IGBLAST_DIR, s, nproc=4)

# Итог
print("\n" + "=" * 60)
print("IgBLAST done:")
for s in SAMPLES:
    ts = Path(IGBLAST_DIR) / f"{s}_igblast.tsv"
    if ts.exists():
        n_lines = int(subprocess.run(
            ["wc", "-l", str(ts)],
            capture_output=True, text=True
        ).stdout.split()[0])
        size_mb = ts.stat().st_size / 1e6
        print(f"  {s}: {n_lines:,} lines, {size_mb:.1f} MB")
    else:
        print(f"  {s}: MISSING")

print("\n=== ERP003950 annotation COMPLETE ===")
